# LC mesh — figures & reproduction

This notebook replaces the original `mesh_and_figure_generation.ipynb`. All real
logic lives in the `lc_mesh` library (`code/lc_mesh/`); cells here are thin calls.

It has two parts:
1. **Figures** — 1:1 reproductions of the paper figures, generated from the
   **canonical published meshes** (the distributed `LC_percentile_meshes` asset),
   exactly as the original notebook did.
2. **Reproduce / explore mesh generation** — regenerate the core mesh from the
   points and quantify how close it is to the published mesh.

On CodeOcean this runs in the capsule environment (`environment/postInstall`,
pinned to the mesh-generation date).

In [1]:
import sys, pathlib, numpy as np
sys.path.insert(0, str(pathlib.Path.cwd()))  # import lc_mesh from code/
import lc_mesh
from lc_mesh import figures

# Resolve data paths: CodeOcean capsule mounts, else the locally downloaded asset.
CAP = pathlib.Path('/root/capsule/data')
if CAP.exists():
    MESH_DIR = CAP / 'LC_percentile_meshes'
    BRAIN_OBJ = CAP / 'ccf_meshes/mcc/annotation/ccf_2017/structure_meshes/structure_997.obj'
    TRAILMAP = CAP / 'LC_H2B_trailmap_probabilities_and_point_calls/segmentation_and_quantification'
    RESULTS = pathlib.Path('/root/capsule/results')
else:
    REPO = pathlib.Path.cwd().parent
    MESH_DIR = REPO / 'results-c712751d-f744-4fe8-9657-93a7084eab22'
    BRAIN_OBJ = None
    TRAILMAP = None
    RESULTS = REPO / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
print('mesh dir:', MESH_DIR)

2026-06-17 22:46:40.615067: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-17 22:46:41.228651: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-17 22:46:43.225931: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2026-06-17 22:46:43.227035: W tensorflow/

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
mesh dir: /root/capsule/data/LC_percentile_meshes


## 1. Load points + canonical published meshes

In [2]:
import trimesh
df = lc_mesh.load_lc_points_csv(MESH_DIR / 'LC_points.csv')
percentile_meshes, core_mesh = lc_mesh.load_published_meshes(MESH_DIR)
brain_mesh = trimesh.load(BRAIN_OBJ) if BRAIN_OBJ and BRAIN_OBJ.exists() else None
# percentile-mesh membership (cell 46); the CSV already carries in_new_core_mesh
df = lc_mesh.count_points_in_meshes(df, percentile_meshes)
print(len(df), 'points;', len(percentile_meshes), 'percentile meshes; brain mesh:', brain_mesh is not None)

34998 points; 9 percentile meshes; brain mesh: True


## 2. Figures (reproduced from the canonical published meshes)
These use the exact distributed meshes, so they match the paper figures.

### 2a. kNN minimum-projection heatmaps

In [ ]:
figures.plot_knn_min_projection(df, save_path=RESULTS / 'kNN_min_projection_heatmaps.png')

### 2b. 3D mesh renderings (percentile + core, all percentiles, point membership)

In [ ]:
figures.plot_percentile_and_core(percentile_meshes, core_mesh, brain_mesh,
                                 save_path=str(RESULTS / 'percentile_90_plus_core.html')).show()
figures.plot_all_percentile_meshes(percentile_meshes, brain_mesh,
                                   save_path=str(RESULTS / 'all_percentiles.html')).show()
figures.plot_points_core_membership(df, percentile_meshes, core_mesh, brain_mesh,
                                    save_path=str(RESULTS / 'LC_points_red_in_core.html')).show()

### 2c. Per-sample point counts in each mesh

In [ ]:
figures.plot_percentile_counts_by_hemisphere(df)

### 2d. Coronal slices with per-sample PC2 histograms

In [ ]:
_ = figures.plot_coronal_slices_with_pc2(df, core_mesh, save_dir=str(RESULTS / 'slice_svgs'))
print('saved coronal slice SVGs')

### 2e. Raw-image max-projection overlay
Needs the SmartSPIM_807324 zarr + raw points (CodeOcean only).

In [ ]:
if TRAILMAP is not None:
    import zarr
    sample = 'SmartSPIM_807324_2025-08-25_11-34-40_stitched_2025-10-23_17-35-23'
    pts = np.load(TRAILMAP / sample / 'points_raw_807324.npy')
    zarr_vol = zarr.open(str(CAP / sample / 'image_tile_fusing/OMEZarr/Ex_488_Em_525.zarr'), mode='r')['1']
    for ap in range(3000, 3100, 100):
        fig, _ = figures.plot_max_proj_with_points(
            zarr_vol, pts, ap_range=(ap, ap + 250), vmax=10000,
            save_path=str(RESULTS / f'max_proj_AP_{ap:04d}_{ap+250:04d}.tiff'))
        fig.show()
else:
    print('raw imagery not mounted locally; run on CodeOcean for this figure')

## 3. Reproduce / explore mesh generation
Regenerate the mesh from points (vs. the canonical mesh used above for figures).

### 3a. kNN-percentile threshold explorer (published core used 67)

In [ ]:
import ipywidgets as widgets
widgets.interact(
    lambda threshold: figures.plot_threshold_scatter(df, threshold),
    threshold=widgets.FloatSlider(value=67, min=0, max=100, step=1, continuous_update=False),
)

### 3b. Regenerate the core mesh and compare to the published one

In [ ]:
mesh, _raw = lc_mesh.make_core_mesh(df, verbose=True)
cmp = lc_mesh.compare_meshes(mesh, core_mesh)
print({k: round(v, 4) if isinstance(v, float) else v
       for k, v in cmp.items() if not isinstance(v, dict)})
figures.plot_mesh_3d(mesh, title='regenerated core mesh')

### 3c. Percentile meshes (generation params recovered; repair steps not)
Uncomment to regenerate. The per-mesh interactive repair steps were never saved,
so these won't match the published percentile meshes exactly.

In [ ]:
# mesh90, _ = lc_mesh.make_percentile_mesh(df, 90, verbose=True)
# figures.plot_mesh_3d(mesh90, title='percentile 90 (generation params only)')

### 3d. Self-registration + self-registered core mesh (SLOW, ~40 min)
Off by default. Reproduces notebook cells 56-68.

In [ ]:
RUN_REGISTRATION = False  # set True to run the ~40 min EMD registration
if RUN_REGISTRATION:
    df_reg, results, ref_key = lc_mesh.self_register(df, verbose=True)
    self_reg_mesh, _ = lc_mesh.make_self_registered_core_mesh(df_reg, verbose=True)
    print('reference group:', ref_key)
    print('self-reg core volume mm3:', round(self_reg_mesh.volume / 1e9, 6))
    figures.plot_mesh_3d(self_reg_mesh, title='self-registered core mesh')